# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, following best practices with Croissant schema references by `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will inspect the structure of the dataset, list all record sets and for each list their fields/columns (all referenced by their `@id`).

In [ ]:
# List all record sets and their fields with @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields/Columns @id in this set:")
    for field in fields:
        print(f"    - {field['@id'] if isinstance(field, dict) and '@id' in field else str(field)}")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract the record set with `@id` `https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034` (the main table), as revealed in the Croissant metadata.

In [ ]:
# Main record set (replace with others if desired)
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
# You can add more record set ids to this list based on overview
record_sets_ids = [main_record_set_id]

dataframes = {}
for rs_id in record_sets_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering out proteins with very low coverage and normalizing the `coverage` numeric field. All field/column references use their Croissant `@id` values.

In [ ]:
# Example field @ids. Please run the 'Data Overview' above to confirm actual field @ids for your case.
# For demonstration, we'll use field @ids that are likely to be present based on mass spec tables:

record_set_id = main_record_set_id
# Please update these if needed based on your specific dataset fields listing
numeric_field_id = 'coverage'  # e.g. field representing protein coverage (%) or similar
group_field_id = 'sample'      # e.g. field representing sample label/group

df = dataframes[record_set_id]
# Ensure numeric field exists and contains numeric data
if numeric_field_id in df.columns:
    # Filter records where 'coverage' > a threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with @{numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized @{numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if present (e.g. sample/group)
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by @{group_field_id} (avg. {numeric_field_id}):")
        display(grouped_df)
    else:
        print(f"Group field @{group_field_id} not found in columns: {df.columns.tolist()}")
else:
    print(f"Numeric field @{numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset, such as the normalized protein coverage.

In [ ]:
import matplotlib.pyplot as plt

# Visualize histogram for the (filtered, normalized) numeric field if present
if 'coverage_normalized' in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df['coverage_normalized'], bins=30, color='steelblue', alpha=0.7)
    plt.title('Normalized Coverage Distribution (>10%)')
    plt.xlabel('Normalized Coverage')
    plt.ylabel('Count')
    plt.grid(True)
    plt.show()

# Boxplot by group (if group_field available)
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset on extracellular vesicle protein mass spectrometry analysis using `mlcroissant`, explored its schema, extracted main records by `@id`, performed basic filtering and normalization steps, and visualized the data. 

This workflow can be extended to support more complex feature engineering, biological interpretation, and integration with downstream ML/analysis pipelines. For reproducible field mapping, always use values referenced by their Croissant `@id`.